# 面试问题：Multimodal RAG 怎样保留 Layout、Region 和 Table Evidence？

可以直接复述的回答是：第一，PDF 解析后每个文本块、表格单元和图注都要有 page、bbox、region_id。第二，普通文本检索适合找章节，表格问题还需链接 row、column 和单位。第三，答案引用应指向具体区域而不只是整页。第四，表格重建要按几何行列排序，并验证合计。第五，页脚单位和脚注必须与表格共同进入上下文。第六，用逐问题数值、区域证据和失败 OCR 对照评估。下面用一份离线季度报告布局演示。

## 真实案例：从季度经营报告回答区域收入问题

报告包含标题、正文、一个 2025 Q2 区域收入表、单位脚注和风险说明。五个问题覆盖单元格、同比、合计、单位和页码。所有区域、数值与 bbox 都是教学构造，不读取真实 PDF 或图片；结构与版面解析输出一致。

In [1]:
regions = [  # 定义带页面、坐标和类型的多模态版面区域
    {"id": "R1", "page": 1, "bbox": (50, 40, 550, 90), "type": "title", "text": "2025 年第二季度经营报告"},  # 文档标题
    {"id": "R2", "page": 1, "bbox": (50, 110, 550, 160), "type": "paragraph", "text": "本季度重点关注区域收入与供应风险。"},  # 正文摘要
    {"id": "T-H1", "page": 2, "bbox": (60, 80, 180, 110), "type": "table_cell", "table": "T1", "row": 0, "col": 0, "text": "区域"},  # 表头区域列
    {"id": "T-H2", "page": 2, "bbox": (180, 80, 330, 110), "type": "table_cell", "table": "T1", "row": 0, "col": 1, "text": "2025 Q2 收入"},  # 表头收入列
    {"id": "T-H3", "page": 2, "bbox": (330, 80, 470, 110), "type": "table_cell", "table": "T1", "row": 0, "col": 2, "text": "同比"},  # 表头同比列
    {"id": "T-N1", "page": 2, "bbox": (60, 110, 180, 140), "type": "table_cell", "table": "T1", "row": 1, "col": 0, "text": "华北"},  # 华北行标签
    {"id": "T-N2", "page": 2, "bbox": (180, 110, 330, 140), "type": "table_cell", "table": "T1", "row": 1, "col": 1, "text": "120"},  # 华北收入数值
    {"id": "T-N3", "page": 2, "bbox": (330, 110, 470, 140), "type": "table_cell", "table": "T1", "row": 1, "col": 2, "text": "+8%"},  # 华北同比
    {"id": "T-S1", "page": 2, "bbox": (60, 140, 180, 170), "type": "table_cell", "table": "T1", "row": 2, "col": 0, "text": "华南"},  # 华南行标签
    {"id": "T-S2", "page": 2, "bbox": (180, 140, 330, 170), "type": "table_cell", "table": "T1", "row": 2, "col": 1, "text": "80"},  # 华南收入数值
    {"id": "T-S3", "page": 2, "bbox": (330, 140, 470, 170), "type": "table_cell", "table": "T1", "row": 2, "col": 2, "text": "-3%"},  # 华南同比
    {"id": "T-T1", "page": 2, "bbox": (60, 170, 180, 200), "type": "table_cell", "table": "T1", "row": 3, "col": 0, "text": "合计"},  # 合计行标签
    {"id": "T-T2", "page": 2, "bbox": (180, 170, 330, 200), "type": "table_cell", "table": "T1", "row": 3, "col": 1, "text": "200"},  # 合计收入数值
    {"id": "R3", "page": 2, "bbox": (60, 210, 470, 235), "type": "footnote", "text": "单位：百万元；收入为未审计口径。"},  # 表格单位与口径脚注
    {"id": "R4", "page": 3, "bbox": (50, 80, 550, 140), "type": "paragraph", "text": "华南同比下降主要受渠道调整影响。"},  # 风险解释正文
]  # 结束版面区域集合
questions = [  # 定义五个需要布局或表格证据的问题
    {"id": "MM-01", "text": "华北 2025 Q2 收入是多少？", "expected": "120百万元", "gold": {"T-N1", "T-N2", "T-H2", "R3"}},  # 行列与单位组合
    {"id": "MM-02", "text": "华南收入同比变化多少？", "expected": "-3%", "gold": {"T-S1", "T-S3", "T-H3"}},  # 表格同比单元格
    {"id": "MM-03", "text": "区域收入合计是多少？", "expected": "200百万元", "gold": {"T-T1", "T-T2", "R3"}},  # 合计与单位
    {"id": "MM-04", "text": "表格金额单位是什么？", "expected": "百万元", "gold": {"R3"}},  # 页脚证据
    {"id": "MM-05", "text": "华南同比下降原因在哪一页？", "expected": "第3页", "gold": {"R4"}},  # 正文页面定位
]  # 结束五个多模态 RAG 问题
print("版面输入：region | page | type | bbox | text")  # 展示解析器保留的空间字段
for region in regions:  # 逐区域输出标题、表格单元和脚注
    print(f"{region['id']} | p{region['page']} | {region['type']:10} | {region['bbox']} | {region['text']}")  # 呈现具体区域证据
print("问题集：", [(item["id"], item["text"], item["expected"]) for item in questions])  # 展示五个布局问答样本


版面输入：region | page | type | bbox | text
R1 | p1 | title      | (50, 40, 550, 90) | 2025 年第二季度经营报告
R2 | p1 | paragraph  | (50, 110, 550, 160) | 本季度重点关注区域收入与供应风险。
T-H1 | p2 | table_cell | (60, 80, 180, 110) | 区域
T-H2 | p2 | table_cell | (180, 80, 330, 110) | 2025 Q2 收入
T-H3 | p2 | table_cell | (330, 80, 470, 110) | 同比
T-N1 | p2 | table_cell | (60, 110, 180, 140) | 华北
T-N2 | p2 | table_cell | (180, 110, 330, 140) | 120
T-N3 | p2 | table_cell | (330, 110, 470, 140) | +8%
T-S1 | p2 | table_cell | (60, 140, 180, 170) | 华南
T-S2 | p2 | table_cell | (180, 140, 330, 170) | 80
T-S3 | p2 | table_cell | (330, 140, 470, 170) | -3%
T-T1 | p2 | table_cell | (60, 170, 180, 200) | 合计
T-T2 | p2 | table_cell | (180, 170, 330, 200) | 200
R3 | p2 | footnote   | (60, 210, 470, 235) | 单位：百万元；收入为未审计口径。
R4 | p3 | paragraph  | (50, 80, 550, 140) | 华南同比下降主要受渠道调整影响。
问题集： [('MM-01', '华北 2025 Q2 收入是多少？', '120百万元'), ('MM-02', '华南收入同比变化多少？', '-3%'), ('MM-03', '区域收入合计是多少？', '200百万元'), ('MM-04', '表格金额单位是什么？', '百万元'), ('

## Baseline / 基线：按页展平 OCR 文本

展平后数字、行名、列名和脚注失去关联。关键词 Top-2 可能找到“华北”和“收入”，却不知道 120 属于哪一列，也常漏掉单位。

In [2]:
page_text = {}  # 收集每页按解析顺序拼接的 OCR 文本
for region in regions:  # 遍历所有版面区域
    page_text.setdefault(region["page"], []).append(region["text"])  # 只按页保存文本而丢失 bbox 和单元格坐标
flat_pages = {page: " ".join(texts) for page, texts in page_text.items()}  # 构造普通文本 RAG 的页级文档
def overlap_score(question, text):  # 使用字符集合重合计算透明文本相关性
    return len(set(question) & set(text))  # 忽略几何和表格结构
baseline_rows = []  # 收集五题页级 Top-1 和证据覆盖
print("展平 Baseline：id | top_page | guessed_answer | evidence_coverage")  # 输出结构丢失的后果
for question in questions:  # 对五个问题执行页级检索
    top_page = max(flat_pages, key=lambda page: overlap_score(question["text"], flat_pages[page]))  # 选择字符重合最高页面
    guessed = "120" if "华北" in question["text"] else ("80" if "华南" in question["text"] else "unknown")  # 模拟从扁平数字附近猜值
    page_regions = {region["id"] for region in regions if region["page"] == top_page}  # 获取整页粗粒度证据
    coverage = len(page_regions & question["gold"]) / len(question["gold"])  # 计算是否覆盖人工需要区域
    baseline_rows.append({"id": question["id"], "page": top_page, "answer": guessed, "coverage": coverage})  # 保存基线结果
    print(f"{question['id']} | {top_page} | {guessed} | {coverage:.0%}")  # 展示猜值和粗粒度引用


展平 Baseline：id | top_page | guessed_answer | evidence_coverage
MM-01 | 2 | 120 | 100%
MM-02 | 2 | 80 | 100%
MM-03 | 2 | unknown | 100%
MM-04 | 2 | unknown | 100%
MM-05 | 3 | 80 | 100%


## 核心实现：按 Row/Column 重建表格并绑定邻近脚注

表格单元由 table、row、col 排序成矩阵。查询解析器识别行名、列名与单位需求，答案保留每个单元格 region_id 和 bbox。

In [3]:
table_cells = [region for region in regions if region["type"] == "table_cell" and region.get("table") == "T1"]  # 提取 T1 的所有结构化单元格
max_row = max(cell["row"] for cell in table_cells)  # 获取表格最大行编号
max_col = max(cell["col"] for cell in table_cells)  # 获取表格最大列编号
table_matrix = [[None for _ in range(max_col + 1)] for _ in range(max_row + 1)]  # 初始化行列矩阵
cell_matrix = [[None for _ in range(max_col + 1)] for _ in range(max_row + 1)]  # 初始化对应区域对象矩阵
for cell in table_cells:  # 按解析器给出的几何行列填充矩阵
    table_matrix[cell["row"]][cell["col"]] = cell["text"]  # 保存可读单元格文本
    cell_matrix[cell["row"]][cell["col"]] = cell  # 保存区域 id、页码和 bbox
header_to_col = {value: index for index, value in enumerate(table_matrix[0])}  # 建立表头到列编号映射
row_to_index = {row[0]: index for index, row in enumerate(table_matrix[1:], start=1)}  # 建立行标签到行编号映射
footnote = next(region for region in regions if region["id"] == "R3")  # 获取表格下方单位脚注
print("重建表格：")  # 输出核心 layout 中间量
for row in table_matrix:  # 逐行展示四行三列表格
    print(row)  # 保留行列对应关系
print("表头映射：", header_to_col, "行映射：", row_to_index)  # 展示查询到坐标的链接


重建表格：
['区域', '2025 Q2 收入', '同比']
['华北', '120', '+8%']
['华南', '80', '-3%']
['合计', '200', None]
表头映射： {'区域': 0, '2025 Q2 收入': 1, '同比': 2} 行映射： {'华北': 1, '华南': 2, '合计': 3}


## 失败案例与修正：OCR 数字异常和单位遗漏

若华北收入 OCR 为 `1,20`，直接字符串解析可能得到 1.2 或失败。表格合计提供约束：华北 + 华南 应等于 200；修正候选 120 后通过。单位脚注必须与数值共同引用。

In [4]:
ocr_value = "1,20"  # 构造华北收入的常见逗号错位 OCR 结果
naive_numeric = float(ocr_value.replace(",", "."))  # 天真把逗号当小数点得到 1.2
south_value = float(table_matrix[row_to_index["华南"]][header_to_col["2025 Q2 收入"]])  # 读取结构化华南收入 80
total_value = float(table_matrix[row_to_index["合计"]][header_to_col["2025 Q2 收入"]])  # 读取表格合计 200
candidate_values = [naive_numeric, float(ocr_value.replace(",", ""))]  # 生成小数和千位分隔两种候选
validated_north = next(value for value in candidate_values if abs(value + south_value - total_value) < 1e-9)  # 使用合计约束选择 120
unit = "百万元" if "百万元" in footnote["text"] else "unknown"  # 从邻近脚注恢复金额单位
print(f"OCR 原值={ocr_value}，天真解析={naive_numeric}")  # 展示错位逗号导致的数值错误
print(f"候选={candidate_values}，华南={south_value}，合计={total_value}，验证后华北={validated_north}")  # 展示表格约束修正
print("单位来源：", footnote["id"], footnote["bbox"], unit)  # 展示页脚区域证据


OCR 原值=1,20，天真解析=1.2
候选=[1.2, 120.0]，华南=80.0，合计=200.0，验证后华北=120.0
单位来源： R3 (60, 210, 470, 235) 百万元


## 结果表：五题的答案与 Region Evidence

In [5]:
def answer_question(question):  # 按问题类型返回答案和精确区域证据
    if question["id"] == "MM-01":  # 回答华北 Q2 收入
        evidence = [cell_matrix[row_to_index["华北"]][0], cell_matrix[row_to_index["华北"]][header_to_col["2025 Q2 收入"]], cell_matrix[0][header_to_col["2025 Q2 收入"]], footnote]  # 组合行、值、列和单位区域
        return f"{table_matrix[row_to_index['华北']][header_to_col['2025 Q2 收入']]}{unit}", evidence  # 返回数值加单位
    if question["id"] == "MM-02":  # 回答华南同比
        evidence = [cell_matrix[row_to_index["华南"]][0], cell_matrix[row_to_index["华南"]][header_to_col["同比"]], cell_matrix[0][header_to_col["同比"]]]  # 组合行、值和列头
        return table_matrix[row_to_index["华南"]][header_to_col["同比"]], evidence  # 返回负三个百分点
    if question["id"] == "MM-03":  # 回答区域收入合计
        evidence = [cell_matrix[row_to_index["合计"]][0], cell_matrix[row_to_index["合计"]][header_to_col["2025 Q2 收入"]], footnote]  # 组合合计行和值及单位
        return f"{table_matrix[row_to_index['合计']][header_to_col['2025 Q2 收入']]}{unit}", evidence  # 返回合计与单位
    if question["id"] == "MM-04":  # 回答金额单位
        return unit, [footnote]  # 只引用具体脚注区域
    cause_region = next(region for region in regions if region["id"] == "R4")  # 定位华南同比下降说明
    return f"第{cause_region['page']}页", [cause_region]  # 返回页面并引用正文区域
layout_rows = []  # 收集五题的答案和区域证据
print("id | expected | answer | evidence(region/page/bbox) | correct")  # 输出逐问题多模态结果
for question in questions:  # 对五题执行 layout-aware 回答
    answer, evidence = answer_question(question)  # 获取结构化答案和证据区域
    evidence_ids = {region["id"] for region in evidence}  # 提取引用区域身份
    correct = answer == question["expected"]  # 与人工答案比较
    layout_rows.append({"id": question["id"], "answer": answer, "evidence": evidence_ids, "correct": correct})  # 保存完整结果
    evidence_view = [(region["id"], region["page"], region["bbox"]) for region in evidence]  # 生成可读空间引用
    print(f"{question['id']} | {question['expected']} | {answer} | {evidence_view} | {correct}")  # 展示答案与精确区域
layout_accuracy = sum(row["correct"] for row in layout_rows) / len(layout_rows)  # 计算五题答案准确率
evidence_recall = sum(len(row["evidence"] & question["gold"]) / len(question["gold"]) for row, question in zip(layout_rows, questions)) / len(questions)  # 计算平均区域证据覆盖率
print(f"Layout RAG：accuracy={layout_accuracy:.1%}，region_evidence_recall={evidence_recall:.1%}")  # 输出答案与引用双指标


id | expected | answer | evidence(region/page/bbox) | correct
MM-01 | 120百万元 | 120百万元 | [('T-N1', 2, (60, 110, 180, 140)), ('T-N2', 2, (180, 110, 330, 140)), ('T-H2', 2, (180, 80, 330, 110)), ('R3', 2, (60, 210, 470, 235))] | True
MM-02 | -3% | -3% | [('T-S1', 2, (60, 140, 180, 170)), ('T-S3', 2, (330, 140, 470, 170)), ('T-H3', 2, (330, 80, 470, 110))] | True
MM-03 | 200百万元 | 200百万元 | [('T-T1', 2, (60, 170, 180, 200)), ('T-T2', 2, (180, 170, 330, 200)), ('R3', 2, (60, 210, 470, 235))] | True
MM-04 | 百万元 | 百万元 | [('R3', 2, (60, 210, 470, 235))] | True
MM-05 | 第3页 | 第3页 | [('R4', 3, (50, 80, 550, 140))] | True
Layout RAG：accuracy=100.0%，region_evidence_recall=100.0%


## 结果解读

MM-01 不是只返回 120，而是同时绑定华北行、收入列、数值单元格和单位脚注四个区域。OCR 反例利用表格合计约束把 1.2 修正为 120。MM-05 引用第 3 页 R4 的具体 bbox，而非整个 PDF。Multimodal RAG 的核心是保留结构证据，而不是把图片 OCR 后当长字符串。

## 生产边界

真实系统需要 PDF 渲染、OCR 置信度、表格检测、合并单元格、阅读顺序、图片与图表理解以及坐标到原文件的可视化高亮。数值校验还需单位换算和会计口径，不能只依赖合计。文档权限和版本也必须绑定 region。本例使用人工布局对象，没有处理扫描噪声。

## 最小回归测试

In [6]:
assert len(regions) >= 5 and len(questions) >= 5  # 保证版面与问题集具有真实教学规模
assert table_matrix[1][1] == "120" and table_matrix[2][2] == "-3%"  # 保证几何行列重建正确
assert naive_numeric == 1.2 and validated_north == 120.0  # 保证 OCR 数值失败可复现并由合计约束修正
assert unit == "百万元"  # 保证金额单位来自脚注区域
assert next(row for row in layout_rows if row["id"] == "MM-01")["evidence"] == questions[0]["gold"]  # 保证华北答案引用行列值和单位
assert layout_accuracy == 1.0 and evidence_recall == 1.0  # 保证五题答案与区域证据完整
